# Exploring different parameter combinations 

In [ ]:

import os, time, json
import itertools
import numpy as np
import pandas as pd 
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components

# ----------------------- General Configuration -----------------------
INPUT_CSV = "6.csv"            # default input path (adjust as needed)
OUTDIR = "results_param_explorer"
SAMPLES_TO_SAVE_PER_CAND = 3   # how many pruned outputs to save per candidate (0 = none)
SAVE_SAMPLED_PRUNED = True     # whether to save pruned files for candidates
N_JOBS = 1

# filter condition for showing results (tunable)
MIN_REMOVED_ABS = 25           # min removed points (e.g. 25)
MAX_REMOVED_FRAC = 0.5         # max removal fraction (e.g. less than half)
# Alternatively, use min/max range conditions.

# ----------------------- Parameter search ranges -----------------------
# (kept ranges small for speed; expand as needed)
grid = {
    "K_DENSITY":        [6, 8, 12],          # k for k-distance computation
    "KEEP_PERCENT":     [82, 86, 90],        # percentile keep threshold for k-distance
    "K_GRAPH":          [4, 7, 10],          # k for kNN graph before MST
    "EDGE_THRESH_VAL":  [95.0, 98.0, 99.5],  # percentile threshold for edge pruning
    "DIST_FACTOR":      [2.5, 4.0, 6.0],     # distance factor for removing far components
    "MIN_COMP_SIZE":    [8, 16, 32],         # min component size to retain
}

# Cap the number of combinations (if needed)
MAX_COMBINATIONS = 200  # if the grid grows too large, reduce this first

# ----------------------- Core pipeline functions (MST-prune) -----------------------
def load_csv_xyz(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, comment='#')
    cols = df.columns.tolist()
    if set(['X','Y','Z']).issubset(set(cols)):
        arr = df[['X','Y','Z']].values
    else:
        arr = df.iloc[:, :3].values
    return arr.astype(float), df

def save_csv_xyz(path, pts, header="X,Y,Z"):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    np.savetxt(path, pts, delimiter=",", header=header, comments='')

def build_knn_graph(points, k=8, n_jobs=1):
    n = len(points)
    if n == 0:
        return csr_matrix((n,n))
    k_use = min(k+1, n)
    nbrs = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree', n_jobs=n_jobs).fit(points)
    dists, idxs = nbrs.kneighbors(points)
    rows, cols, data = [], [], []
    for i in range(n):
        for j in range(1, idxs.shape[1]):
            rows.append(i); cols.append(int(idxs[i, j])); data.append(float(dists[i, j]))
    A = csr_matrix((data, (rows, cols)), shape=(n, n))
    A = (A + A.T) / 2.0
    return A

def mst_edges_from_adj(A):
    mst = minimum_spanning_tree(A)
    mst = mst.tocoo()
    edges = list(zip(mst.row.tolist(), mst.col.tolist(), mst.data.tolist()))
    normalized = []
    for u,v,w in edges:
        if u <= v:
            normalized.append((int(u), int(v), float(w)))
        else:
            normalized.append((int(v), int(u), float(w)))
    normalized = list({(u,v,w) for (u,v,w) in normalized})
    return normalized

def cut_long_edges_and_components(points, k_graph=12, edge_thresh_val=95.0, dist_factor=6.0, min_comp_size=10, n_jobs=1):
    info = {}
    n = len(points)
    if n == 0:
        return np.zeros(0, dtype=bool), info
    A = build_knn_graph(points, k=k_graph, n_jobs=n_jobs)
    edges = mst_edges_from_adj(A)
    if len(edges) == 0:
        return np.ones(n, dtype=bool), info
    weights = np.array([w for (_,_,w) in edges])
    info['mst_edge_count'] = int(len(weights))
    thresh = float(np.percentile(weights, edge_thresh_val))
    info['edge_thresh'] = thresh
    rows, cols, data = [], [], []
    for (u,v,w) in edges:
        if w <= thresh:
            rows.append(u); cols.append(v); data.append(w)
            rows.append(v); cols.append(u); data.append(w)
    if len(rows) == 0:
        labels = np.arange(n)
        sizes = np.bincount(labels, minlength=n)
        info['pruned_edge_count'] = 0
    else:
        B = csr_matrix((data, (rows, cols)), shape=(n, n))
        n_comp, labels = connected_components(B, directed=False, connection='weak')
        sizes = np.bincount(labels)
        info['pruned_edge_count'] = int(len(data)//2)
        info['n_components'] = int(n_comp)
    # centroids
    centroids = np.zeros((len(sizes), 3))
    for i in range(len(sizes)):
        idx = np.where(labels == i)[0]
        if len(idx) > 0:
            centroids[i] = points[idx].mean(axis=0)
        else:
            centroids[i] = np.array([np.nan, np.nan, np.nan])
    # find large cores
    large_idxs = np.where(sizes >= (min_comp_size*2))[0]
    if len(large_idxs)==0:
        order = np.argsort(sizes)[::-1]
        large_idxs = order[:1]
    info['large_idxs'] = [int(x) for x in large_idxs.tolist()]
    # scale = median NN
    nn = NearestNeighbors(n_neighbors=min(2, max(2, n)), algorithm='kd_tree').fit(points)
    dists, _ = nn.kneighbors(points)
    median_nn = float(np.median(dists[:, -1]))
    info['median_nn'] = median_nn
    dist_thresh = dist_factor * median_nn
    info['dist_thresh'] = dist_thresh
    # decide keep per component
    keep_comp = np.ones(len(sizes), dtype=bool)
    for comp in range(len(sizes)):
        if sizes[comp] >= min_comp_size:
            keep_comp[comp] = True
            continue
        if len(large_idxs)>0:
            dists_to_large = np.linalg.norm(centroids[large_idxs] - centroids[comp], axis=1)
            dmin = float(np.min(dists_to_large))
        else:
            dmin = float('inf')
        if dmin > dist_thresh:
            keep_comp[comp] = False
        else:
            keep_comp[comp] = True
    mask_keep = np.array([keep_comp[lab] for lab in labels], dtype=bool)
    info['component_sizes'] = [int(x) for x in sizes.tolist()]
    info['kept_components'] = int(np.sum(keep_comp))
    info['total_components'] = int(len(sizes))
    return mask_keep, info

def run_prune_once(points, params, n_jobs=1):
    """
    points: ndarray Nx3
    params: dict with keys K_DENSITY, KEEP_PERCENT, K_GRAPH, EDGE_THRESH_VAL, DIST_FACTOR, MIN_COMP_SIZE
    returns: dict of stats and pruned_points
    """
    n0 = len(points)
    out = {'params': params.copy(), 'n_input': n0}
    # density-step (k-distance)
    k_density = int(params['K_DENSITY'])
    keep_percent = float(params['KEEP_PERCENT'])
    k_use = min(k_density+1, max(2, n0))
    nn = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree', n_jobs=n_jobs).fit(points)
    dists, _ = nn.kneighbors(points)
    kth = dists[:, -1]
    kth_thresh = float(np.percentile(kth, keep_percent))
    mask_density = kth <= kth_thresh
    pts_den = points[mask_density]
    out['n_after_density'] = int(len(pts_den))
    # prune
    mask_keep, info = cut_long_edges_and_components(pts_den,
                                                    k_graph=int(params['K_GRAPH']),
                                                    edge_thresh_val=float(params['EDGE_THRESH_VAL']),
                                                    dist_factor=float(params['DIST_FACTOR']),
                                                    min_comp_size=int(params['MIN_COMP_SIZE']),
                                                    n_jobs=n_jobs)
    pts_final = pts_den[mask_keep]
    out.update({
        'n_final': int(len(pts_final)),
        'n_removed': int(n0 - len(pts_final)),
        'kth_thresh': kth_thresh,
        'prune_info': info
    })
    return out, pts_final

# ----------------------- grid generator helper -----------------------
def gen_param_combinations(grid):
    keys = list(grid.keys())
    values = [grid[k] for k in keys]
    combos = list(itertools.product(*values))
    param_dicts = [dict(zip(keys, combo)) for combo in combos]
    # limit combos if requested
    if len(param_dicts) > MAX_COMBINATIONS:
        print(f"[INFO] combination count ({len(param_dicts)}) is large - capping at {MAX_COMBINATIONS} (random sample).")
        rng = np.random.default_rng(0)
        idxs = rng.choice(len(param_dicts), size=MAX_COMBINATIONS, replace=False)
        param_dicts = [param_dicts[i] for i in idxs]
    return param_dicts

# ----------------------- Main explorer -----------------------
def explore_and_save(input_csv, outdir, grid, min_removed_abs=MIN_REMOVED_ABS, max_removed_frac=MAX_REMOVED_FRAC,
                     save_samples=SAVE_SAMPLED_PRUNED, samples_to_save=SAMPLES_TO_SAVE_PER_CAND, n_jobs=1):
    pts, df = load_csv_xyz(input_csv)
    total_n = len(pts)
    combos = gen_param_combinations(grid)
    print(f"[RUN] total points={total_n}, exploring {len(combos)} parameter combos ...")
    results = []
    os.makedirs(outdir, exist_ok=True)
    summary_rows = []
    t0_all = time.time()
    for i, params in enumerate(combos):
        t0 = time.time()
        try:
            stats, pts_final = run_prune_once(pts, params, n_jobs=n_jobs)
        except Exception as e:
            print(f"[ERR] combo {i} failed: {e}")
            continue
        stats['run_time_s'] = time.time() - t0
        stats['combo_idx'] = i
        # flatten some info for CSV
        pi = stats.pop('prune_info', {})
        row = {
            **{f'p_{k}': v for k,v in params.items()},
            'combo_idx': i,
            'n_input': stats['n_input'],
            'n_after_density': stats['n_after_density'],
            'n_final': stats['n_final'],
            'n_removed': stats['n_removed'],
            'run_time_s': stats['run_time_s'],
            'kth_thresh': stats.get('kth_thresh', np.nan),
            'median_nn': pi.get('median_nn', np.nan),
            'edge_thresh': pi.get('edge_thresh', np.nan),
            'n_components': pi.get('n_components', np.nan),
            'kept_components': pi.get('kept_components', np.nan),
        }
        summary_rows.append(row)
        results.append((params, stats, pts_final))
        if (i+1) % 10 == 0 or (i+1) == len(combos):
            print(f"[PROG] {i+1}/{len(combos)} combos done.")
    t_all = time.time()-t0_all
    print(f"[DONE] exploration finished in {t_all:.1f}s. Saving summary ...")
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(outdir, "results_summary.csv")
    summary_df.to_csv(summary_csv, index=False)
    # filter candidates
    candidates = summary_df[(summary_df['n_removed'] >= min_removed_abs) & (summary_df['n_removed'] < (max_removed_frac * total_n))]
    candidates = candidates.sort_values('n_removed', ascending=False).reset_index(drop=True)
    cand_csv = os.path.join(outdir, "candidates.csv")
    candidates.to_csv(cand_csv, index=False)
    print(f"[SAVE] wrote summary ({len(summary_df)} rows) -> {summary_csv}")
    print(f"[SAVE] wrote candidates ({len(candidates)} rows) -> {cand_csv}")
    # optionally save some pruned sample files for top candidates
    if save_samples and len(candidates) > 0:
        saved = 0
        for idx, row in candidates.iterrows():
            if saved >= samples_to_save:
                break
            combo_idx = int(row['combo_idx'])
            params, stats, pts_final = results[combo_idx]
            fname = f"pruned_combo_{combo_idx:03d}_removed_{int(row['n_removed'])}.csv"
            fpath = os.path.join(outdir, "samples", fname)
            save_csv_xyz(fpath, pts_final)
            saved += 1
        print(f"[SAVE] saved {saved} sample pruned outputs in {os.path.join(outdir, 'samples')}")
    return summary_csv, cand_csv

# ----------------------- CLI / run -----------------------
if __name__ == "__main__":
    print("=== prune_param_explorer: starting ===")
    print("INPUT:", INPUT_CSV)
    summary_csv, cand_csv = explore_and_save(INPUT_CSV, OUTDIR, grid,
                                             min_removed_abs=MIN_REMOVED_ABS,
                                             max_removed_frac=MAX_REMOVED_FRAC,
                                             save_samples=SAVE_SAMPLED_PRUNED,
                                             samples_to_save=SAMPLES_TO_SAVE_PER_CAND,
                                             n_jobs=N_JOBS)
    print("Finished. Summary:", summary_csv)
    print("Candidates:", cand_csv)


In [ ]:
# pip install --upgrade pandas


# Imports and initial configuration

In [ ]:

import os, time, json
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components

SUMMARY_CSV = "results_summary.csv"   # file containing parameter rows
INPUT_CSV = "6.csv"                   # main input file (X,Y,Z)
OUTDIR = "results_parsa"              # output directory
OUTNAME_PREFIX = "from_summary_row6"   # output file name prefix
N_JOBS = 1
# 1-based row index 
ROW = 7

## ----------------- IO helper functions -----------------


In [ ]:

def load_csv_xyz(path):
    df = pd.read_csv(path, comment='#')
    cols = df.columns.tolist()
    if set(['X','Y','Z']).issubset(set(cols)):
        arr = df[['X','Y','Z']].values
    else:
        arr = df.iloc[:, :3].values
    return arr.astype(float), df

def save_csv_xyz(path, pts, header="X,Y,Z"):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    np.savetxt(path, pts, delimiter=",", header=header, comments='')


## ----------------- kNN graph and MST -----------------


In [ ]:

def build_knn_graph(points, k=8, n_jobs=1):
    n = len(points)
    if n == 0:
        return csr_matrix((n,n))
    k_use = min(k+1, n)
    nbrs = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree', n_jobs=n_jobs).fit(points)
    dists, idxs = nbrs.kneighbors(points)
    rows, cols, data = [], [], []
    for i in range(n):
        for j in range(1, idxs.shape[1]):
            rows.append(i); cols.append(int(idxs[i, j])); data.append(float(dists[i, j]))
    A = csr_matrix((data, (rows, cols)), shape=(n, n))
    A = (A + A.T) / 2.0
    return A

def mst_edges_from_adj(A):
    if A.nnz == 0:
        return []
    mst = minimum_spanning_tree(A)
    mst = mst.tocoo()
    edges = list(zip(mst.row.tolist(), mst.col.tolist(), mst.data.tolist()))
    normalized = []
    for u,v,w in edges:
        u=int(u); v=int(v); w=float(w)
        if u < v:
            normalized.append((u,v,w))
        else:
            normalized.append((v,u,w))
    normalized = list({(u,v,w) for (u,v,w) in normalized})
    return normalized



## ----------------- Cut long edges and remove small components -----------------


In [ ]:
def cut_long_edges_and_components(points, k_graph=12, edge_thresh_mode='percentile', edge_thresh_val=95.0,
                                  dist_factor=6.0, min_comp_size=10, large_comp_min=30, n_jobs=1):
    info = {}
    n = len(points)
    if n == 0:
        return np.zeros(0, dtype=bool), info

    A = build_knn_graph(points, k=k_graph, n_jobs=n_jobs)
    edges = mst_edges_from_adj(A)
    if len(edges) == 0:
        return np.ones(n, dtype=bool), info
    weights = np.array([w for (_,_,w) in edges], dtype=float)
    info['mst_edge_count'] = int(len(weights))
    if edge_thresh_mode == 'percentile':
        thresh = float(np.percentile(weights, edge_thresh_val))
    elif edge_thresh_mode == 'median_std':
        thresh = float(np.median(weights) + edge_thresh_val * np.std(weights))
    else:
        thresh = float(edge_thresh_val)
    info['edge_thresh'] = thresh

    rows, cols, data = [], [], []
    for (u,v,w) in edges:
        if w <= thresh:
            rows.append(u); cols.append(v); data.append(w)
            rows.append(v); cols.append(u); data.append(w)
    if len(rows) == 0:
        labels = np.arange(n)
        sizes = np.bincount(labels, minlength=n)
        info['pruned_edge_count'] = 0
    else:
        B = csr_matrix((data, (rows, cols)), shape=(n, n))
        n_comp, labels = connected_components(B, directed=False, connection='weak')
        sizes = np.bincount(labels)
        info['pruned_edge_count'] = int(len(data)//2)
        info['n_components'] = int(n_comp)

    centroids = np.zeros((len(sizes), 3))
    for i in range(len(sizes)):
        idx = np.where(labels == i)[0]
        if len(idx) > 0:
            centroids[i] = points[idx].mean(axis=0)
        else:
            centroids[i] = np.array([np.nan, np.nan, np.nan])
    large_idxs = np.where(sizes >= large_comp_min)[0]
    if len(large_idxs) == 0:
        order = np.argsort(sizes)[::-1]
        large_idxs = order[:2] if len(order)>1 else order[:1]
    info['large_idxs'] = [int(x) for x in large_idxs.tolist()]

    nn = NearestNeighbors(n_neighbors=min(2, max(2, n)), algorithm='kd_tree').fit(points)
    dists, _ = nn.kneighbors(points)
    median_nn = float(np.median(dists[:, -1]))
    info['median_nn'] = median_nn
    dist_thresh = dist_factor * median_nn
    info['dist_thresh'] = dist_thresh

    keep_comp = np.ones(len(sizes), dtype=bool)
    for comp in range(len(sizes)):
        if sizes[comp] >= min_comp_size:
            keep_comp[comp] = True
            continue
        if len(large_idxs)>0:
            dists_to_large = np.linalg.norm(centroids[large_idxs] - centroids[comp], axis=1)
            dmin = float(np.min(dists_to_large))
        else:
            dmin = float('inf')
        if dmin > dist_thresh:
            keep_comp[comp] = False
        else:
            keep_comp[comp] = True
    mask_keep = np.array([keep_comp[lab] for lab in labels], dtype=bool)
    info['component_sizes'] = [int(x) for x in sizes.tolist()]
    info['kept_components'] = int(np.sum(keep_comp))
    info['total_components'] = int(len(sizes))
    return mask_keep, info


## ----------------- Main pipeline function with parameters -----------------


In [ ]:

def run_prune_pipeline(input_csv, outdir, outname,
                       k_density=8, keep_percent=88.0,
                       k_graph=7, edge_thresh_mode='percentile', edge_thresh_val=98.0,
                       dist_factor=4.0, min_comp_size=15, large_comp_min=30, n_jobs=1):
    t0 = time.time()
    pts, df_orig = load_csv_xyz(input_csv)
    n0 = len(pts)
    print(f"[RUN] loaded {n0} pts from {input_csv}")
    # density keep
    k_use = min(k_density+1, max(2, n0))
    nn = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree', n_jobs=n_jobs).fit(pts)
    dists, _ = nn.kneighbors(pts)
    kth = dists[:, -1]
    kth_thresh = float(np.percentile(kth, keep_percent))
    mask_density = kth <= kth_thresh
    pts_den = pts[mask_density]
    print(f"[RUN] density: kept {len(pts_den)} / {n0}  (keep_percent={keep_percent}, kth-thresh={kth_thresh:.6g})")
    if len(pts_den) == 0:
        print("[ERROR] All points removed by density filter. Relax parameters.")
        stats = {'n_input': n0, 'n_after_density': 0, 'n_final': 0, 'n_removed': n0, 'run_time_s': time.time()-t0,
                 'kth_thresh': kth_thresh}
        return None, None, stats

    mask_keep, info = cut_long_edges_and_components(pts_den,
                                                    k_graph=k_graph,
                                                    edge_thresh_mode=edge_thresh_mode,
                                                    edge_thresh_val=edge_thresh_val,
                                                    dist_factor=dist_factor,
                                                    min_comp_size=min_comp_size,
                                                    large_comp_min=large_comp_min,
                                                    n_jobs=n_jobs)
    pts_final = pts_den[mask_keep]
    t1 = time.time()
    stats = {
        'n_input': n0,
        'n_after_density': int(np.sum(mask_density)),
        'n_final': len(pts_final),
        'n_removed': n0 - len(pts_final),
        'run_time_s': t1 - t0,
        'kth_thresh': kth_thresh,
        'median_nn': info.get('median_nn', None),
        'edge_thresh': info.get('edge_thresh', None),
        'n_components': info.get('n_components', None),
        'kept_components': info.get('kept_components', None),
        'prune_info': info
    }
    os.makedirs(outdir, exist_ok=True)
    outpath = os.path.join(outdir, outname)
    if pts_final is not None:
        save_csv_xyz(outpath, pts_final)
        print("[RUN] saved final ->", outpath)
    return outpath, pts_final, stats
 

## ------------ Wrapper: read row from summary and run pipeline -------------


In [ ]:

def apply_row_from_summary(summary_csv, row_1based, input_csv, outdir, outname_prefix, n_jobs=1):
    if not os.path.exists(summary_csv):
        raise FileNotFoundError(f"Summary CSV not found: {summary_csv}")
    df = pd.read_csv(summary_csv)
    if row_1based < 1 or row_1based > len(df):
        raise IndexError(f"Row {row_1based} out of range (1..{len(df)})")
    
    
    row = df.iloc[row_1based - 1]    # the row whose parameters we want to run
    
    
    
    mapping = {
        'p_K_DENSITY': 'k_density',
        'p_KEEP_PERCENT': 'keep_percent',
        'p_K_GRAPH': 'k_graph',
        'p_EDGE_THRESH_VAL': 'edge_thresh_val',
        'p_DIST_FACTOR': 'dist_factor',
        'p_MIN_COMP_SIZE': 'min_comp_size'
    }
    
    # default values when column is missing
    
    defaults = {'k_density':8, 'keep_percent':88.0, 'k_graph':7, 'edge_thresh_val':98.0,
                'dist_factor':4.0, 'min_comp_size':15}
    params = {}
    for col, pname in mapping.items():
        if col in row.index:
            val = row[col]
            try:
                # attempt numeric conversion
                if pd.isna(val):
                    params[pname] = defaults[pname]
                else:
                    params[pname] = float(val) if (isinstance(val, (int,float)) or str(val).replace('.','',1).isdigit()) else val
            except Exception:
                params[pname] = val
        else:
            params[pname] = defaults[pname]
    # some parameters must be int
    params['k_density'] = int(params.get('k_density', defaults['k_density']))
    params['k_graph'] = int(params.get('k_graph', defaults['k_graph']))
    params['min_comp_size'] = int(params.get('min_comp_size', defaults['min_comp_size']))
    params['keep_percent'] = float(params.get('keep_percent', defaults['keep_percent']))
    params['edge_thresh_val'] = float(params.get('edge_thresh_val', defaults['edge_thresh_val']))
    params['dist_factor'] = float(params.get('dist_factor', defaults['dist_factor']))

    # output name based on row index
    outname = f"{outname_prefix}_row{row_1based}.csv"
    print("Using parameters from summary row", row_1based, "->", params)
    outpath, pts_final, stats = run_prune_pipeline(
        input_csv=input_csv, outdir=outdir, outname=outname,
        k_density=params['k_density'], keep_percent=params['keep_percent'],
        k_graph=params['k_graph'], edge_thresh_mode='percentile', edge_thresh_val=params['edge_thresh_val'],
        dist_factor=params['dist_factor'], min_comp_size=params['min_comp_size'], large_comp_min=30, n_jobs=n_jobs
    )
    # print readable stats
    if stats is not None:
        print("\n=== SUMMARY ===")
        print(f"initial_count: {stats['n_input']}")
        print(f"after_density_count: {stats['n_after_density']}")
        print(f"final_count: {stats['n_final']}")
        print(f"n_removed: {stats['n_removed']}")
        print(f"run_time_s: {stats['run_time_s']:.3f}")
        print(f"kth_thresh: {stats.get('kth_thresh')}")
        print(f"median_nn: {stats.get('median_nn')}")
        print(f"edge_thresh: {stats.get('edge_thresh')}")
        print(f"n_components: {stats.get('n_components')}")
        print(f"kept_components: {stats.get('kept_components')}")
    return outpath, pts_final, stats

# ----------------- Run wrapper for specified ROW -----------------
print("Applying row", ROW, "from", SUMMARY_CSV)
outpath, pts_final, stats = apply_row_from_summary(SUMMARY_CSV, ROW, INPUT_CSV, OUTDIR, OUTNAME_PREFIX, n_jobs=N_JOBS)
print("Done. final saved at:", outpath)
